In [8]:
import os
import json
import requests

import pandas as pd
import dotenv
import redis

In [9]:
# open .env file and get API keys
env_path = os.path.abspath('../.env.development.local')
dotenv.load_dotenv(env_path)
KV_REST_API_READ_ONLY_TOKEN = os.getenv("KV_REST_API_READ_ONLY_TOKEN")
KV_REST_API_TOKEN = os.getenv("KV_REST_API_TOKEN")
KV_REST_API_URL = os.getenv("KV_REST_API_URL")
KV_URL = os.getenv("KV_URL")

# Set headers for authentication
headers = {
    "Authorization": f"Bearer {KV_REST_API_TOKEN}",
    "Content-Type": "application/json"
}

In [10]:
# Adjust url to work with redis
redis_url = KV_URL
if redis_url.startswith("redis://"):
    redis_url = 'rediss://' + redis_url[len('redis://'):]
r = redis.from_url(redis_url)

In [20]:
# Import datasets
datasets = {}
dataset_names = ['clfever', 'phemeplus', 'vitc']
for dataset_name in dataset_names:
    with open(f'{dataset_name}.json') as f:
        datasets[dataset_name] = json.load(f)

In [23]:
# Populate Vercel KV with datasets
for dataset_name in dataset_names:
    dataset = datasets[dataset_name]
    for datapoint in dataset:
        id = datapoint['claim_id']
        r.hset(id, mapping={
            'claim': datapoint['claim'],
            'evidence': datapoint['evidence'],
            'label': datapoint['label']
        })

In [21]:
# Create batches containing 25 datapoints each
batch_ids = []
for dataset_name in dataset_names:
    dataset = datasets[dataset_name]
    for i in range(0, len(dataset), 25):
        batch_dataset = dataset[i:i+25]
        claim_ids = []
        for batch_datapoint in batch_dataset:
            id = batch_datapoint['claim_id']
            claim_ids.append(id)
        batch_id = f'batch_{dataset_name}_{i//25 + 1}'
        batch_ids.append(batch_id)
        # r.hset(batch_id, mapping={'claim_ids': json.dumps(claim_ids)})    

In [42]:
# Create queue 
r.lpush('queue', *batch_ids)

50

In [32]:
queue = r.lrange('queue', 0, -1)
print(len(queue))
print(queue)

36
[b'batch_clfever_1', b'batch_vitc_2', b'batch_vitc_1', b'batch_phemeplus_4', b'batch_phemeplus_3', b'batch_phemeplus_2', b'batch_phemeplus_1', b'batch_clfever_1', b'batch_vitc_2', b'batch_vitc_1', b'batch_phemeplus_4', b'batch_phemeplus_3', b'batch_phemeplus_2', b'batch_phemeplus_1', b'batch_clfever_1', b'batch_vitc_2', b'batch_vitc_1', b'batch_phemeplus_4', b'batch_phemeplus_3', b'batch_phemeplus_2', b'batch_phemeplus_1', b'batch_clfever_1', b'batch_vitc_2', b'batch_vitc_1', b'batch_phemeplus_4', b'batch_phemeplus_3', b'batch_phemeplus_2', b'batch_phemeplus_1', b'batch_clfever_1', b'batch_vitc_2', b'batch_vitc_1', b'batch_phemeplus_4', b'batch_phemeplus_3', b'batch_phemeplus_2', b'batch_phemeplus_1', b'batch_clfever_1']


In [40]:
cursor, keys = r.scan(cursor=0)
keys

[b'Sebastian',
 b'batch_clfever_1',
 b'batch_phemeplus_1',
 b'batch_phemeplus_2',
 b'batch_phemeplus_3',
 b'batch_phemeplus_4',
 b'batch_vitc_1',
 b'batch_vitc_2',
 b'cfever_0',
 b'cfever_1']

In [35]:
r.lrange('participants',0,-1)

[b'{"participant":"Sebastian","batchId":"batch_phemeplus_3"}']